# **Model Pool**

Collects the models described by public data on energy, quality and monetary cost (197 candidates), links the sources through the Hugging Face id and the architecture fingerprint, and saves the wide table and the core pool of nine Llama 3, 3.1 and 3.3 models used in the thesis.

In the core pool, energy comes only from ML.ENERGY (H100, vLLM) and quality only from the Open LLM Leaderboard, with MMLU-Pro per subject from its details; a fine-tuned model without measurements inherits the energy of a measured model with the same architecture. Prices are from OpenRouter (snapshot of 20 October 2026); core models without a provider inherit their price in the routing pipeline.

## Install

In [1]:
!pip install -q "mlenergy-data==0.4.0" "datasets>=3.0" huggingface_hub pyarrow requests

## Imports and configuration

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download

POOL_VERSION = "v1.3"
OUTPUT_DIR = "data/model_pool"
KWH_TO_J = 3.6e6

# ML.ENERGY: GPU, task priority (chat first) and the two batch sizes kept
MLENERGY_GPU = "H100"
MLENERGY_TASKS = ["lm-arena-chat", "gpqa", "sourcegraph-fim"]   # image and video tasks are excluded
MLENERGY_BATCHES = [128, 8]   # 128: reference batch of the pipeline; 8: closest to the batch-1 sources
MLENERGY_REF_BATCH = MLENERGY_BATCHES[0]   # configuration (GPUs, precision) whose full batch curve is saved

LLMPERF_REPO = "optimum-benchmark/llm-perf-leaderboard"
LLMPERF_FILES = ["perf-df-unquantized-1xA100.csv", "data/perf-df-pytorch-cuda-unquantized-1xA100.csv"]

AIENERGY_REPO = "AIEnergyScore/Leaderboard"
AIENERGY_FILE = "data/energy/text_generation.csv"

OLL_REPO = "open-llm-leaderboard/contents"
TIGER_REPO = "TIGER-Lab/mmlu_pro_leaderboard_submission"
TIGER_FILE = "results.csv"
EVALPLUS_URL = "https://raw.githubusercontent.com/evalplus/evalplus.github.io/main/results.json"
OPENROUTER_URL = "https://openrouter.ai/api/v1/models"

# GPU memory used to estimate how many GPUs a model needs
GPU_MEMORY_GB = 80          # H100 and A100 80GB
WEIGHT_SHARE = 0.75         # share of GPU memory for the weights; the rest is left for the KV cache

# inspected but not loaded yet (format to be checked first, see 'Inspection of the software-development paper data')
SOFTWARE_PAPER_ZIP = "https://zenodo.org/records/14064267/files/Replication_package.zip?download=1"

### Which models are members of the pool

Chat or instruction-tuned models from the providers listed below, plus a curated list of specialists; merges and amateur fine-tunes are excluded. Base models are not members but can donate their measured energy.

In [ ]:
OFFICIAL_ORGS = {
    "meta-llama", "qwen", "google", "microsoft", "mistralai", "nvidia", "deepseek-ai", "openai",
    "ibm-granite", "huggingfacetb", "allenai", "01-ai", "tiiuae", "bigcode", "codellama",
    "cohereforai", "internlm", "thudm", "zai-org", "upstage", "lgai-exaone", "tencent", "swiss-ai",
}
INSTRUCT_RE = re.compile(r"instruct|chat|-it\b|-it-|thinking|gpt-oss", re.IGNORECASE)
EXCLUDE_RE = re.compile(r"abliterat|uncensor|merge|-gguf|-awq|-gptq|-bnb|\bvl\b|-vl-|omni", re.IGNORECASE)

# model id -> specialization. ids that do not exist on the Hub simply get no config and are reported by the analysis
SPECIALISTS = {
    # mathematics
    "qwen/qwen2.5-math-1.5b-instruct": "mathematics",
    "qwen/qwen2.5-math-7b-instruct": "mathematics",
    "qwen/qwen2.5-math-72b-instruct": "mathematics",
    "qwen/qwen2-math-72b-instruct": "mathematics",
    "nvidia/acemath-1.5b-instruct": "mathematics",
    "nvidia/acemath-7b-instruct": "mathematics",
    "nvidia/acemath-72b-instruct": "mathematics",
    "nvidia/openmath2-llama3.1-8b": "mathematics",
    "deepseek-ai/deepseek-math-7b-instruct": "mathematics",
    # coding
    "qwen/qwen2.5-coder-0.5b-instruct": "coding",
    "qwen/qwen2.5-coder-1.5b-instruct": "coding",
    "qwen/qwen2.5-coder-3b-instruct": "coding",
    "qwen/qwen2.5-coder-7b-instruct": "coding",
    "qwen/qwen2.5-coder-14b-instruct": "coding",
    "qwen/qwen2.5-coder-32b-instruct": "coding",
    "qwen/qwen3-coder-30b-a3b-instruct": "coding",
    "01-ai/yi-coder-9b-chat": "coding",
    "deepseek-ai/deepseek-coder-1.3b-instruct": "coding",
    "deepseek-ai/deepseek-coder-6.7b-instruct": "coding",
    "bigcode/starcoder2-15b-instruct-v0.1": "coding",
    "codellama/codellama-7b-instruct-hf": "coding",
    "google/codegemma-7b-it": "coding",
    # medicine
    "aaditya/llama3-openbiollm-8b": "medicine",
    "aaditya/llama3-openbiollm-70b": "medicine",
    "biomistral/biomistral-7b": "medicine",
    "google/medgemma-4b-it": "medicine",
    "google/medgemma-27b-text-it": "medicine",
    "epfl-llm/meditron-7b": "medicine",
    "m42-health/llama3-med42-70b": "medicine",
    "m42-health/llama3-med42-8b": "medicine",
    "mistralai/mathstral-7b-v0.1": "mathematics",
    # law and finance
    "equall/saul-7b-instruct-v1": "law",
    "sufe-aiflm-lab/fin-r1": "finance",
    # reasoning models (a mode of use rather than a subject, kept as its own label)
    "openai/gpt-oss-20b": "reasoning",
    "openai/gpt-oss-120b": "reasoning",
    "qwen/qwen3-30b-a3b-thinking-2507": "reasoning",
    "qwen/qwen3-235b-a22b-thinking-2507": "reasoning",
    "qwen/qwq-32b": "reasoning",
}

# generalists added by hand because their name has no "instruct"/"chat" (they have MMLU-Pro scores)
EXTRA_MEMBERS = {"microsoft/phi-4"}

# repositories renamed by their provider: old id -> current id
RENAMED = {
    "meta-llama/meta-llama-3.1-8b-instruct": "meta-llama/llama-3.1-8b-instruct",
    "meta-llama/meta-llama-3.1-70b-instruct": "meta-llama/llama-3.1-70b-instruct",
    "meta-llama/meta-llama-3.1-405b-instruct": "meta-llama/llama-3.1-405b-instruct",
    "meta-llama/meta-llama-3.1-8b": "meta-llama/llama-3.1-8b",
    "meta-llama/meta-llama-3.1-70b": "meta-llama/llama-3.1-70b",
    "meta-llama/meta-llama-3.1-405b": "meta-llama/llama-3.1-405b",
}

# core pool: one energy source (ML.ENERGY) and one quality protocol (Open LLM Leaderboard) for all
CORE_POOL = {
    "meta-llama/llama-3.1-8b-instruct": "generalist",
    "meta-llama/meta-llama-3-8b-instruct": "generalist",
    "meta-llama/llama-3.1-70b-instruct": "generalist",
    "meta-llama/llama-3.3-70b-instruct": "generalist",
    "meta-llama/meta-llama-3-70b-instruct": "generalist",
    "nvidia/llama-3.1-nemotron-70b-instruct-hf": "generalist",
    "nvidia/openmath2-llama3.1-8b": "mathematics",
    "aaditya/llama3-openbiollm-70b": "medicine",
    "m42-health/llama3-med42-70b": "medicine",
}
# optional core members: added only if every value the pipeline needs is available (checked in 'Core pool and consistency checks').
CORE_OPTIONAL = {
    "aaditya/llama3-openbiollm-8b": "medicine",
    "m42-health/llama3-med42-8b": "medicine",
}
CORE_CANDIDATES = {**CORE_POOL, **CORE_OPTIONAL}

def key(model_id):
    # common model key: lower-cased Hugging Face id without spaces, with renamed ids mapped to the current one.
    k = str(model_id).strip().lower()
    return RENAMED.get(k, k)

def is_member_candidate(model_id):
    k = key(model_id)
    if k in SPECIALISTS or k in EXTRA_MEMBERS or k in CORE_CANDIDATES:
        return True
    org = k.split("/")[0]
    return org in OFFICIAL_ORGS and bool(INSTRUCT_RE.search(k)) and not EXCLUDE_RE.search(k)

## Hugging Face token

In [4]:
try:
    from google.colab import userdata                 # Colab: secret HF_TOKEN
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:                                    # local run: environment variable or prompt
    from getpass import getpass
    os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN") or getpass("Hugging Face token: ")

HF_TOKEN = os.environ["HF_TOKEN"]
api = HfApi(token=HF_TOKEN)
SNAPSHOT_DIR = os.path.join(OUTPUT_DIR, f"snapshots_{POOL_VERSION}")
os.makedirs(SNAPSHOT_DIR, exist_ok=True)
print("Output folder:", OUTPUT_DIR)

Mounted at /content/drive
Output folder: /content/drive/MyDrive/thesis/model_pool


## Energy source: ML.ENERGY v3

For each model, the task with the highest priority is used (chat, then GPQA, then code completion); at each batch size, the configuration with the lowest energy per output token is kept, bfloat16 preferred over fp8.

In [5]:
from mlenergy.data.records import LLMRuns

mle_raw = LLMRuns.from_hf().to_dataframe()
mle = mle_raw[(mle_raw["gpu_model"] == MLENERGY_GPU)
              & mle_raw["task"].isin(MLENERGY_TASKS)
              & mle_raw["is_stable"].astype(bool)].copy()
mle["task_rank"] = mle["task"].map({t: i for i, t in enumerate(MLENERGY_TASKS)})
mle["precision_rank"] = (mle["weight_precision"] != "bfloat16").astype(int)

def mlenergy_at_batch(batch):
    d = mle[mle["max_num_seqs"] == batch]
    d = d.sort_values(["model_id", "task_rank", "precision_rank", "energy_per_token_joules"])
    d = d.groupby("model_id").head(1)
    return d.set_index("model_id")[[
        "task", "weight_precision", "num_gpus", "energy_per_token_joules", "avg_power_watts",
        "output_throughput_tokens_per_sec", "avg_output_len", "total_params_billions",
        "activated_params_billions", "architecture",
    ]]

mle_by_batch = {b: mlenergy_at_batch(b) for b in MLENERGY_BATCHES}
mlenergy = mle_by_batch[MLENERGY_BATCHES[0]].add_prefix("mle_").join(
    mle_by_batch[MLENERGY_BATCHES[1]][["energy_per_token_joules", "num_gpus"]].add_prefix("mle_b8_"),
    how="outer")
mlenergy = mlenergy.rename(columns={
    "mle_energy_per_token_joules": "mle_j_per_out_token",
    "mle_b8_energy_per_token_joules": "mle_b8_j_per_out_token",
})
mlenergy.index = mlenergy.index.map(key)
print(f"ML.ENERGY: {len(mlenergy)} models on {MLENERGY_GPU}")
print(mlenergy[["mle_task", "mle_num_gpus", "mle_j_per_out_token", "mle_b8_j_per_out_token"]].to_string())

runs/llm.parquet: reconstructing file:   0%|          |  0.00B /  232kB            

runs/llm.parquet: downloading bytes:           |  0.00B            

ML.ENERGY: 22 models on H100
                                                          mle_task  mle_num_gpus  mle_j_per_out_token  mle_b8_j_per_out_token
model_id                                                                                                                     
qwen/qwen3-14b                                       lm-arena-chat           1.0             0.127858                0.706093
qwen/qwen3-235b-a22b-instruct-2507                   lm-arena-chat           8.0             1.486239                7.292589
qwen/qwen3-235b-a22b-instruct-2507-fp8               lm-arena-chat           4.0             1.100829                9.471371
qwen/qwen3-235b-a22b-thinking-2507                             NaN           NaN                  NaN                7.195367
qwen/qwen3-235b-a22b-thinking-2507-fp8                        gpqa           8.0             1.384629                9.022119
qwen/qwen3-30b-a3b-instruct-2507                     lm-arena-chat           2.0         

## ML.ENERGY batch curve

Every batch size measured with the configuration kept at the reference batch, so that energy and throughput at any batch capacity describe the same deployment.

In [6]:
ref_cfg = mle_by_batch[MLENERGY_REF_BATCH]
curve_parts = []
for model_id, r in ref_cfg.iterrows():
    curve_parts.append(mle[(mle["model_id"] == model_id) & (mle["task"] == r["task"])
                           & (mle["num_gpus"] == r["num_gpus"]) & (mle["weight_precision"] == r["weight_precision"])])
curve_cols = {"max_num_seqs": "batch", "energy_per_token_joules": "j_per_out_token",
              "output_throughput_tokens_per_sec": "throughput_tok_s", "avg_power_watts": "power_w",
              "mean_itl_ms": "itl_ms", "avg_batch_size": "avg_batch", "avg_output_len": "avg_output_len"}
mle_curve = pd.concat(curve_parts)
mle_curve = (mle_curve.assign(model_key=mle_curve["model_id"].map(key))
             .groupby(["model_key", "task", "num_gpus", "weight_precision", "max_num_seqs"], as_index=False)
             [[c for c in curve_cols if c in mle_curve]].mean()
             .rename(columns=curve_cols).sort_values(["model_key", "batch"]))
mle_curve["batch"] = mle_curve["batch"].astype(int)
missing_curve_cols = [v for c, v in curve_cols.items() if v not in mle_curve]
if missing_curve_cols:
    print("Not in this ML.ENERGY release:", missing_curve_cols)
print(f"Batch curve: {mle_curve['model_key'].nunique()} models, {len(mle_curve)} rows")
show_keys = [m for m in mle_curve["model_key"].unique() if m.startswith("meta-llama/llama-3")]
print(mle_curve[mle_curve["model_key"].isin(show_keys)].round(3).to_string(index=False))

Batch curve: 21 models, 195 rows
                             model_key          task  num_gpus weight_precision  batch  j_per_out_token  throughput_tok_s  power_w  itl_ms  avg_batch  avg_output_len
meta-llama/llama-3.1-405b-instruct-fp8 lm-arena-chat         8              fp8      8           13.975           233.307 3260.433  34.060      7.981         365.660
meta-llama/llama-3.1-405b-instruct-fp8 lm-arena-chat         8              fp8     16            7.589           445.929 3384.258  35.653     15.948         380.203
meta-llama/llama-3.1-405b-instruct-fp8 lm-arena-chat         8              fp8     32            4.765           785.742 3744.285  40.389     31.915         372.102
meta-llama/llama-3.1-405b-instruct-fp8 lm-arena-chat         8              fp8     64            3.082          1350.240 4162.080  46.894     63.864         359.689
meta-llama/llama-3.1-405b-instruct-fp8 lm-arena-chat         8              fp8     96            2.661          1681.587 4474.554  55.60

## Energy source: LLM-Perf Leaderboard

Batch 1; energy per token is the GPU energy of the prefill or decode phase divided by its tokens. Wide table only.

In [7]:
perf_parts = []
for f in LLMPERF_FILES:
    try:
        perf_parts.append(pd.read_csv(hf_hub_download(LLMPERF_REPO, f, repo_type="dataset", token=HF_TOKEN)))
    except Exception as e:
        print(f"skipped {f}: {e}")
perf = pd.concat(perf_parts, ignore_index=True)
if "report.traceback" in perf:
    perf = perf[perf["report.traceback"].isna()]
perf = perf.dropna(subset=["report.prefill.energy.gpu", "report.decode.energy.gpu"])

batch = perf["config.scenario.input_shapes.batch_size"]
n_in = perf["config.scenario.input_shapes.sequence_length"] * batch
n_out = perf["config.scenario.generate_kwargs.max_new_tokens"] * batch
perf = perf.assign(
    model=perf["config.backend.model"].map(key),
    llmperf_j_per_in_token=perf["report.prefill.energy.gpu"] * KWH_TO_J / n_in,
    llmperf_j_per_out_token=perf["report.decode.energy.gpu"] * KWH_TO_J / n_out,
    llmperf_j_per_out_token_total=perf["report.decode.energy.total"] * KWH_TO_J / n_out,
    llmperf_dtype=perf["config.backend.torch_dtype"],
    llmperf_attn=perf["config.backend.attn_implementation"],
    llmperf_in_tokens=perf["config.scenario.input_shapes.sequence_length"],
    llmperf_out_tokens=perf["config.scenario.generate_kwargs.max_new_tokens"],
)
llmperf = (perf.sort_values("llmperf_j_per_out_token").groupby("model").head(1)
               .set_index("model")[[c for c in perf.columns if c.startswith("llmperf_")]])
print(f"LLM-Perf: {len(llmperf)} models (A100, batch 1)")
print(llmperf.head().to_string())

perf-df-unquantized-1xA100.csv:   0%|          | 0.00/2.97M [00:00<?, ?B/s]

(…)f-df-pytorch-cuda-unquantized-1xA100.csv:   0%|          | 0.00/2.99M [00:00<?, ?B/s]

LLM-Perf: 35 models (A100, batch 1)
                         llmperf_j_per_in_token  llmperf_j_per_out_token  llmperf_j_per_out_token_total llmperf_dtype       llmperf_attn  llmperf_in_tokens  llmperf_out_tokens
model                                                                                                                                                                          
facebook/opt-125m                      0.003624                 0.692412                       2.374492       float16  flash_attention_2                256                  64
eleutherai/gpt-neo-125m                0.004250                 0.832497                       2.872585       float16  flash_attention_2                256                  64
openai-community/gpt2                  0.004543                 0.913051                       2.922463       float16  flash_attention_2                256                  64
facebook/opt-350m                      0.006543                 1.289774            

## Energy source: AI Energy Score

The February and December 2025 campaigns differ by more than an order of magnitude for the same architecture, so they are kept as two separate sources. `total_gpu_energy` is in kWh per 1,000 queries. Wide table only.

In [8]:
aies = pd.read_csv(hf_hub_download(AIENERGY_REPO, AIENERGY_FILE, repo_type="space", token=HF_TOKEN))
aies["mode"] = aies["model"].str.extract(r"\s+(low|medium|high)$", expand=False)
aies["model_key"] = aies["model"].str.replace(r"\s+(low|medium|high)$", "", regex=True).str.replace("*", "", regex=False).map(key)
aies = aies[aies["mode"].isna() | (aies["mode"] == "medium")]
aienergy = aies.drop_duplicates("model_key").set_index("model_key").rename(columns={
    "total_gpu_energy": "aies_total_gpu_energy", "energy_score": "aies_star_rating",
    "class": "aies_class", "test date": "aies_test_date", "mode": "aies_mode",
})[["aies_total_gpu_energy", "aies_star_rating", "aies_class", "aies_test_date", "aies_mode"]]
aienergy["aies_wh_per_1k_queries"] = aienergy["aies_total_gpu_energy"] * 1000
aienergy["aies_j_per_query"] = aienergy["aies_total_gpu_energy"] * KWH_TO_J / 1000
aienergy["aies_campaign"] = aienergy["aies_test_date"].map({"Feb 25": "feb25", "Dec 25": "dec25"})
print(aienergy["aies_campaign"].value_counts(dropna=False).to_string())
assert aienergy["aies_campaign"].notna().all(), "unknown AI Energy Score test date"
aies_by_campaign = {c: d.drop(columns="aies_campaign").rename(columns=lambda col: col.replace("aies_", f"aies_{c}_"))
                    for c, d in aienergy.groupby("aies_campaign")}
print(f"AI Energy Score: {len(aienergy)} text-generation models")
print(aienergy.head().to_string())

text_generation.csv:   0%|          | 0.00/2.79k [00:00<?, ?B/s]

aies_campaign
feb25    40
dec25    22
AI Energy Score: 62 text-generation models
                                    aies_total_gpu_energy  aies_star_rating aies_class aies_test_date aies_mode  aies_wh_per_1k_queries  aies_j_per_query aies_campaign
model_key                                                                                                                                                              
berkeley-nest/starling-lm-7b-alpha               0.019011                 2          A         Feb 25       NaN                  19.011           68.4396         feb25
distilbert/distilgpt2                            0.001313                 5          A         Feb 25       NaN                   1.313            4.7268         feb25
salesforce/xlam-7b-r                             0.019170                 2          A         Feb 25       NaN                  19.170           69.0120         feb25
openai-community/gpt2                            0.002146                 5    

## Quality source: Open LLM Leaderboard v2

Normalized scores (a random answer scores zero) and raw accuracies are both kept; for repeated evaluations, the most recent is kept.

In [9]:
oll_raw = load_dataset(OLL_REPO, split="train").to_pandas()
BENCH = ["IFEval", "BBH", "MATH Lvl 5", "GPQA", "MUSR", "MMLU-PRO"]
oll = oll_raw[~oll_raw["Flagged"].astype(bool) & oll_raw["Available on the hub"].astype(bool)].copy()
oll["model_key"] = oll["fullname"].map(key)
oll = oll.sort_values("Submission Date").groupby("model_key").tail(1)

rename = {}
for b in BENCH:
    slug = re.sub(r"[^a-z0-9]+", "_", b.lower()).strip("_")
    rename[b] = f"oll_{slug}"
    rename[f"{b} Raw"] = f"oll_{slug}_raw"
rename.update({"#Params (B)": "oll_params_b", "Type": "oll_type", "Base Model": "oll_base_model",
               "Official Providers": "oll_official", "Submission Date": "oll_eval_date",
               "MoE": "oll_moe", "Architecture": "oll_architecture"})
openllm = oll.set_index("model_key").rename(columns=rename)[list(rename.values())]
print(f"Open LLM Leaderboard: {len(openllm)} models after filtering")

README.md:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4576 [00:00<?, ? examples/s]

Open LLM Leaderboard: 2831 models after filtering


## Quality source: EvalPlus (code)

Wide table only.

In [10]:
evalplus_json = requests.get(EVALPLUS_URL, timeout=60).json()
with open(os.path.join(SNAPSHOT_DIR, "evalplus_results.json"), "w") as f:
    json.dump(evalplus_json, f)

rows = []
for name, e in evalplus_json.items():
    link = e.get("link") or ""
    m = re.search(r"huggingface\.co/([^/\s]+/[^/\s?#]+)", link)
    if not m:
        continue
    p = e.get("pass@1", {})
    rows.append({"model_key": key(m.group(1)), "evalplus_name": name,
                 "evalplus_humaneval": p.get("humaneval"), "evalplus_humaneval_plus": p.get("humaneval+"),
                 "evalplus_mbpp": p.get("mbpp"), "evalplus_mbpp_plus": p.get("mbpp+")})
evalplus = pd.DataFrame(rows).drop_duplicates("model_key").set_index("model_key")
print(f"EvalPlus: {len(evalplus)} models with a Hugging Face link")

EvalPlus: 82 models with a Hugging Face link


## Cost source: OpenRouter prices

USD per million tokens; free variants are ignored and the cheapest paid endpoint is kept.

In [11]:
try:
    or_json = requests.get(OPENROUTER_URL, timeout=60).json()
    with open(os.path.join(SNAPSHOT_DIR, "openrouter_models.json"), "w") as f:
        json.dump(or_json, f)
    price_snapshot_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")
except Exception as e:
    print("OpenRouter not reachable:", e)
    or_json, price_snapshot_utc = {"data": []}, None

rows = []
for m in or_json.get("data", []):
    hf = m.get("hugging_face_id")
    pr = m.get("pricing", {}) or {}
    if not hf or str(m.get("id", "")).endswith(":free"):
        continue
    try:
        p_in, p_out = float(pr.get("prompt", 0)) * 1e6, float(pr.get("completion", 0)) * 1e6
    except (TypeError, ValueError):
        continue
    if p_in <= 0 and p_out <= 0:
        continue
    rows.append({"model_key": key(hf), "price_openrouter_id": m.get("id"),
                 "price_in_usd_per_mtok": p_in, "price_out_usd_per_mtok": p_out,
                 "openrouter_context_length": m.get("context_length")})
prices = (pd.DataFrame(rows, columns=["model_key", "price_openrouter_id", "price_in_usd_per_mtok",
                                      "price_out_usd_per_mtok", "openrouter_context_length"])
            .sort_values("price_out_usd_per_mtok").drop_duplicates("model_key").set_index("model_key"))
print(f"OpenRouter: {len(prices)} priced models with a Hugging Face id")

OpenRouter: 154 priced models with a Hugging Face id


## Members and metadata from the Hub

`config.json` gives the architecture fingerprint and the context window, the safetensors metadata the parameter count and `weights_gb`. `gpus_needed_est` is an estimate used only in the wide table; the core pool uses `mle_num_gpus`, the number of GPUs used by ML.ENERGY.

In [ ]:
# every model measured by ML.ENERGY on a chat, reasoning or code task is served as a chat model
candidates = set(SPECIALISTS) | EXTRA_MEMBERS | set(CORE_CANDIDATES) | set(mlenergy.index)
for idx in (mlenergy.index, llmperf.index, aienergy.index, openllm.index, evalplus.index, prices.index):
    candidates |= {k for k in idx if is_member_candidate(k)}
members = sorted(candidates)
donors = sorted(set(mlenergy.index) | set(llmperf.index) | set(aienergy.index))
print(f"Members: {len(members)} | energy-measured models: {len(donors)}")

DTYPE_BYTES = {"F64": 8, "F32": 4, "BF16": 2, "F16": 2, "F8_E4M3": 1, "F8_E5M2": 1,
               "I64": 8, "I32": 4, "I16": 2, "I8": 1, "U8": 1, "BOOL": 1}

ARCH_KEYS = ["model_type", "num_hidden_layers", "hidden_size", "intermediate_size",
             "num_attention_heads", "num_key_value_heads", "num_local_experts", "num_experts",
             "num_experts_per_tok", "moe_intermediate_size"]

def fetch_metadata(model_id):
    out = {"model_key": key(model_id), "config_ok": False}
    try:
        with open(hf_hub_download(model_id, "config.json", token=HF_TOKEN)) as f:
            cfg = json.load(f)
        text_cfg = {**cfg, **cfg.get("text_config", {})}          # multimodal configs nest the text model
        arch = {k: text_cfg.get(k) for k in ARCH_KEYS}
        arch["model_type"] = cfg.get("model_type")
        out.update(config_ok=True,
                   arch_fingerprint=json.dumps(arch, sort_keys=True),
                   model_type=cfg.get("model_type"),
                   context_window=text_cfg.get("max_position_embeddings"),
                   hidden_size=text_cfg.get("hidden_size"),
                   num_hidden_layers=text_cfg.get("num_hidden_layers"))
    except Exception as e:
        out["config_error"] = type(e).__name__
    try:
        meta = api.get_safetensors_metadata(model_id)
        out["params_b_hub"] = sum(meta.parameter_count.values()) / 1e9
        out["weights_gb"] = sum(n * DTYPE_BYTES.get(dt, 2) for dt, n in meta.parameter_count.items()) / 1e9
        out["weights_dtypes"] = ",".join(sorted(meta.parameter_count))
    except Exception:
        out["params_b_hub"] = np.nan
    return out

meta_rows = []
to_fetch = sorted(set(members) | set(donors))
for i, m in enumerate(to_fetch):
    meta_rows.append(fetch_metadata(m))
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(to_fetch)}")
META_COLS = ["config_ok", "config_error", "arch_fingerprint", "model_type", "context_window",
             "hidden_size", "num_hidden_layers", "params_b_hub", "weights_gb", "weights_dtypes"]
hub_meta = pd.DataFrame(meta_rows).set_index("model_key").reindex(columns=META_COLS)
hub_meta["config_ok"] = hub_meta["config_ok"].fillna(False).astype(bool)
print(f"config.json read for {int(hub_meta['config_ok'].sum())} of {len(hub_meta)} models")
gated = hub_meta.index[hub_meta["config_error"] == "GatedRepoError"]
print(f"Gated repositories (accept the licence on these pages with the account of HF_TOKEN, then re-run):")
for m in gated:
    print(f"  https://huggingface.co/{m}   {'member' if m in members else 'energy donor'}")
other_errors = hub_meta[hub_meta["config_error"].notna() & (hub_meta["config_error"] != "GatedRepoError")]
print("Other errors (e.g. id not on the Hub):")
print(other_errors[["config_error"]].to_string())
print(hub_meta.loc[~hub_meta["config_ok"], "config_error"].value_counts().to_string())

Members: 197 | energy-measured models: 115


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/44.8k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/15 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/44.8k [00:00<?, ?B/s]

... (700 outputs left out)

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/4.03M [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/93 [00:00<?, ?it/s]

config.json read for 280 of 281 models
Gated repositories (accept the licence on these pages with the account of HF_TOKEN, then re-run):
Other errors (e.g. id not on the Hub):
                                config_error
model_key                                   
crm-sf/textbase-70b  RepositoryNotFoundError
config_error
RepositoryNotFoundError    1


## Linking energy: direct measurement or inheritance by architecture

A member measured itself has **direct** energy; otherwise it **inherits** the energy of a measured model with the same fingerprint, and the donor is recorded (instruction-tuned donors first, then alphabetical order).

In [13]:
fingerprint = hub_meta["arch_fingerprint"]

def link_energy(source_df, prefix):
    cols = list(source_df.columns)
    measured = source_df.join(fingerprint, how="left")
    by_fp = {}
    for donor, row in measured.dropna(subset=["arch_fingerprint"]).iterrows():
        by_fp.setdefault(row["arch_fingerprint"], []).append(donor)
    for fp in by_fp:
        by_fp[fp].sort(key=lambda d: (not bool(INSTRUCT_RE.search(d)), d))
    rows = []
    for m in members:
        if m in source_df.index:
            rows.append({"model_key": m, f"{prefix}_link": "direct", f"{prefix}_donor": m,
                         **source_df.loc[m, cols].to_dict()})
            continue
        fp = fingerprint.get(m)
        donor = by_fp.get(fp, [None])[0] if isinstance(fp, str) else None
        if donor:
            rows.append({"model_key": m, f"{prefix}_link": "inherited", f"{prefix}_donor": donor,
                         **source_df.loc[donor, cols].to_dict()})
        else:
            rows.append({"model_key": m, f"{prefix}_link": None, f"{prefix}_donor": None})
    return pd.DataFrame(rows).set_index("model_key")

energy_links = {
    "mle": link_energy(mlenergy, "mle"),
    "llmperf": link_energy(llmperf, "llmperf"),
    **{f"aies_{c}": link_energy(d, f"aies_{c}") for c, d in aies_by_campaign.items()},
}
ENERGY_SOURCES = list(energy_links)
print(pd.DataFrame({src: df[f"{src}_link"].value_counts(dropna=False) for src, df in energy_links.items()}).fillna(0).astype(int).to_string())

           mle  llmperf  aies_dec25  aies_feb25
direct      22        0           4           5
inherited   12       12          26          55
None       163      185         167         137


## Quality per MMLU-Pro subject (MMLU-Pro leaderboard, TIGER-Lab)

5-shot chain-of-thought, a protocol different from the Open LLM Leaderboard, so the two scores are stored in separate columns. Wide table only.

In [ ]:
TIGER_SUBJECTS = ["Biology", "Business", "Chemistry", "Computer Science", "Economics", "Engineering", "Health",
                  "History", "Law", "Math", "Philosophy", "Physics", "Psychology", "Other"]

# Leaderboard name (normalized) -> HF key, for names the automatic rule cannot match
TIGER_ALIASES = {
    "phi3medium4k": "microsoft/phi-3-medium-4k-instruct",
    "phi3medium128k": "microsoft/phi-3-medium-128k-instruct",
    "phi3mini4k": "microsoft/phi-3-mini-4k-instruct",
    "phi3mini128k": "microsoft/phi-3-mini-128k-instruct",
    "deepseekmath7binstruct": "deepseek-ai/deepseek-math-7b-instruct",
    "qwen272bchat": "qwen/qwen2-72b-instruct",
    "gptoss20b": "openai/gpt-oss-20b",
    "gptoss120b": "openai/gpt-oss-120b",
    "llama4maverick": "meta-llama/llama-4-maverick-17b-128e-instruct",
    "llama4scout": "meta-llama/llama-4-scout-17b-16e-instruct",
}

def norm_name(name):
    n = str(name).lower().split("/")[-1]
    n = re.sub(r"^meta-", "", n)
    n = re.sub(r"-hf$", "", n)
    return re.sub(r"[^a-z0-9]", "", n)

tiger_raw = pd.read_csv(hf_hub_download(TIGER_REPO, TIGER_FILE, repo_type="dataset", token=HF_TOKEN))
t = tiger_raw.copy()
t["variant"] = t["Models"].str.extract(r"\(([^)]*)\)\s*$", expand=False)
t["base_name"] = t["Models"].str.replace(r"\s*\([^)]*\)\s*$", "", regex=True)
t["source"] = t["Data Source"].str.strip().str.lower().map(
    lambda x: "TIGER-Lab" if x.startswith("tiger") else "Self-Reported")
for c in ["Overall"] + TIGER_SUBJECTS:
    t[c] = pd.to_numeric(t[c], errors="coerce")          # "-" -> missing
t["n_subjects"] = t[TIGER_SUBJECTS].notna().sum(axis=1)

members_by_norm = {}
for m in members:
    members_by_norm.setdefault(norm_name(m), []).append(m)
def tiger_keys(name):
    # every member whose normalized name matches (several ids can share one name)
    n = norm_name(name)
    return [TIGER_ALIASES[n]] if n in TIGER_ALIASES else members_by_norm.get(n, [])

t["model_key"] = t["base_name"].map(tiger_keys)
unmatched = sorted(t.loc[t["model_key"].str.len() == 0, "Models"])
t = t.explode("model_key").dropna(subset=["model_key"])
t = (t.assign(_pref=list(zip(t["source"] != "TIGER-Lab", t["n_subjects"] == 0, t["variant"].fillna("") != "medium")))
      .sort_values("_pref").drop_duplicates("model_key").set_index("model_key"))
t = t[t.index.isin(members)]
mmlupro_tiger = t[["Models", "source", "variant", "Overall"] + TIGER_SUBJECTS].copy()
mmlupro_tiger.columns = (["mmlupro_tiger_name", "mmlupro_tiger_source", "mmlupro_tiger_variant", "mmlupro_tiger_overall"]
                         + ["mmlupro_tiger_" + re.sub(r"[^a-z0-9]+", "_", c.lower()) for c in TIGER_SUBJECTS])
print(f"MMLU-Pro (TIGER-Lab): {len(tiger_raw)} rows, {len(mmlupro_tiger)} matched to members, "
      f"{int((t['n_subjects'] > 0).sum())} of them with subject scores")
print(f"{len(unmatched)} leaderboard names not matched to a member (closed models, base models, or names to alias):")
print(", ".join(unmatched))
print(mmlupro_tiger[["mmlupro_tiger_source", "mmlupro_tiger_overall", "mmlupro_tiger_health",
               "mmlupro_tiger_law", "mmlupro_tiger_math"]].sort_values("mmlupro_tiger_overall", ascending=False).head(15).to_string())

results.csv:   0%|          | 0.00/27.5k [00:00<?, ?B/s]

MMLU-Pro (TIGER-Lab): 262 rows, 57 matched to members, 46 of them with subject scores
205 leaderboard names not matched to a member (closed models, base models, or names to alias):
Athene-V2-Chat, Athene-V2-Chat (0-shot), Aya-Expanse-32B, Aya-Expanse-8B, Claude-3-5-Haiku-20241022, Claude-3-Haiku-20240307, Claude-3-Opus, Claude-3-Sonnet, Claude-3.5-Sonnet (2024-06-20), Claude-3.5-Sonnet (2024-10-22), Claude-3.5-Sonnet (2024-10-22), Claude-3.7-Sonnet-Thinking, Claude-4-Opus-Thinking, Claude-4-Sonnet, Claude-4.5-Opus(Thinking), Claude-4.5-Sonnet(Thinking), Claude-4.6-Opus(Thinking), Claude-4.6-Sonnet(Thinking), Cohere-Aya-Vision, DeepSeek-Chat-V2_5, DeepSeek-Coder-V2-Lite-Base, DeepSeek-Coder-V2-Lite-Instruct, DeepSeek-R1, DeepSeek-R1-0528, DeepSeek-V3.1-NonThinking, DeepSeek-V3.1-Thinking, DeepSeek-V3.2-Thinking, Deepseek-V2-Chat, Deepseek-V3, Deepseek-V3-0324, Doubao-1.5-Pro, ECHO_Ego_v2_14B, ERNIE-4.5-21B-A3B-Base, ERNIE-4.5-300B-A47B, ERNIE-4.5-300B-A47B-Base, GLM-4-9B, GLM-4.5, GLM-4

## MMLU-Pro per subject from the Open LLM Leaderboard details (core candidates only)

Per-subject accuracy is the mean of the per-question `acc`. Each details dataset is gated: accept its conditions once on its page.

In [ ]:
oll_fullname = oll.set_index("model_key")["fullname"]

def details_repo(fullname):
    return "open-llm-leaderboard/" + fullname.replace("/", "__") + "-details"

def mmlu_pro_by_subject(fullname):
    # reads the raw per-question file of MMLU-Pro (samples_leaderboard_mmlu_pro_<date>.jsonl)
    repo = details_repo(fullname)
    files = api.list_repo_files(repo, repo_type="dataset")
    runs = sorted(f for f in files if "samples_leaderboard_mmlu_pro" in f)
    if not runs:
        raise FileNotFoundError(f"no MMLU-Pro samples file in {repo}; first files: {files[:8]}")
    d = pd.read_json(hf_hub_download(repo, runs[-1], repo_type="dataset", token=HF_TOKEN), lines=True)
    doc = d["doc"].map(lambda x: json.loads(x) if isinstance(x, str) else x)
    subject = doc.map(lambda x: x.get("category") if isinstance(x, dict) else None)
    acc_col = "acc" if "acc" in d else [c for c in d.columns if c.startswith("acc")][0]
    by_subject = pd.Series(d[acc_col].astype(float).values, index=subject.values).groupby(level=0).mean()
    return by_subject, list(d.columns)

from huggingface_hub import whoami
print("Token account:", whoami(token=HF_TOKEN)["name"])
oll_subject_rows, details_missing = {}, []
for m in CORE_CANDIDATES:
    if m not in oll_fullname.index:
        details_missing.append((m, "not in the Open LLM Leaderboard"))
        continue
    try:
        s, cols = mmlu_pro_by_subject(oll_fullname[m])
        if not oll_subject_rows:
            print("Columns of the details:", cols)
            print("Subjects:", list(s.index))
        oll_subject_rows[m] = s
    except Exception as e:
        details_missing.append((m, f"{type(e).__name__}: {str(e)[:150]}"))
mmlupro_oll = pd.DataFrame(oll_subject_rows).T
mmlupro_oll.columns = ["mmlupro_oll_" + re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_") for c in mmlupro_oll.columns]
print(f"MMLU-Pro per subject (Open LLM details): {len(mmlupro_oll)} of {len(CORE_CANDIDATES)} core candidates")
if details_missing:
    print("Not loaded (accept the conditions on these pages, then run this cell again):")
    for m, why in details_missing:
        link = f"https://huggingface.co/datasets/{details_repo(oll_fullname[m])}" if m in oll_fullname.index else ""
        print(f"  {m}: {why}  {link}")
if len(mmlupro_oll):
    print((mmlupro_oll * 100).round(1).to_string())

Token account: giselleduran


meta-llama__Llama-3.1-8B-Instruct/sample(…): reconstructing file:   0%|          |  0.00B /  327MB            

meta-llama__Llama-3.1-8B-Instruct/sample(…): downloading bytes:           |  0.00B            

Columns of the details: ['doc_id', 'doc', 'target', 'arguments', 'resps', 'filtered_resps', 'doc_hash', 'prompt_hash', 'target_hash', 'acc']
Subjects: ['biology', 'business', 'chemistry', 'computer science', 'economics', 'engineering', 'health', 'history', 'law', 'math', 'other', 'philosophy', 'physics', 'psychology']


meta-llama__Meta-Llama-3-8B-Instruct/sam(…): reconstructing file:   0%|          |  0.00B /  327MB            

meta-llama__Meta-Llama-3-8B-Instruct/sam(…): downloading bytes:           |  0.00B            

... (11 outputs left out)

aaditya__Llama3-OpenBioLLM-70B/samples_l(…): downloading bytes:           |  0.00B            

m42-health__Llama3-Med42-70B/samples_lea(…): reconstructing file:   0%|          |  0.00B /  401MB            

m42-health__Llama3-Med42-70B/samples_lea(…): downloading bytes:           |  0.00B            

MMLU-Pro per subject (Open LLM details): 9 of 11 core candidates
Not loaded (accept the conditions on these pages, then run this cell again):
  aaditya/llama3-openbiollm-8b: not in the Open LLM Leaderboard  
  m42-health/llama3-med42-8b: not in the Open LLM Leaderboard  
                                           mmlupro_oll_biology  mmlupro_oll_business  mmlupro_oll_chemistry  mmlupro_oll_computer_science  mmlupro_oll_economics  mmlupro_oll_engineering  mmlupro_oll_health  mmlupro_oll_history  mmlupro_oll_law  mmlupro_oll_math  mmlupro_oll_other  mmlupro_oll_philosophy  mmlupro_oll_physics  mmlupro_oll_psychology
meta-llama/llama-3.1-8b-instruct                          67.1                  28.0                   27.9                          39.3                   49.4                     32.9                52.0                 45.7             33.6              24.5               37.1                    38.3                 27.8                    57.6
meta-llama/meta-llama-3-8b-i

## Inspection of the software-development paper data

Not used in the thesis.

In [16]:
import io
import zipfile

try:
    blob = requests.get(SOFTWARE_PAPER_ZIP, timeout=120).content
    with open(os.path.join(SNAPSHOT_DIR, "software_paper_replication.zip"), "wb") as f:
        f.write(blob)
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for info in z.infolist()[:80]:
            print(f"{info.file_size:>10,d}  {info.filename}")
except Exception as e:
    print("Not downloaded:", e)

         0  Replication_package/
     2,401  Replication_package/Readme.md
     9,257  Replication_package/environment.yml
         0  Replication_package/data/
   264,119  Replication_package/data/A100_vertical.png
   168,557  Replication_package/data/RTX3070-plots.ipynb
     4,542  Replication_package/data/label_color_dict.json
   795,956  Replication_package/data/A100-plots.ipynb
         0  Replication_package/data/xlsx/
    49,751  Replication_package/data/xlsx/A100.xlsx
    28,671  Replication_package/data/xlsx/RTX3070.xlsx
    27,420  Replication_package/appendix.html
         0  Replication_package/src/
   556,935  Replication_package/src/humanevalpack_extra_docstring_assertion_example.jsonl
       429  Replication_package/src/idle_rapl.py
     1,565  Replication_package/src/cluster_models.txt
     2,522  Replication_package/src/model_specification.ipynb
     1,191  Replication_package/src/warm_up.py
     3,288  Replication_package/src/main_get_completion.py
     1,245  Replica

## Assemble the model table

In [ ]:
def specialisation(m):
    if m in SPECIALISTS:
        return SPECIALISTS[m], "curated"
    if re.search(r"coder|code", m):
        return "coding", "name_rule"
    if re.search(r"math", m):
        return "mathematics", "name_rule"
    if re.search(r"bio|med(?!ium)", m):      # "medium" (e.g. Phi-3-medium) is a size, not medicine
        return "medicine", "name_rule"
    return "generalist", "default"

pool = pd.DataFrame(index=pd.Index(members, name="model_key"))
pool["org"] = pool.index.str.split("/").str[0]
spec = pool.index.map(specialisation)
pool["specialisation"] = [s for s, _ in spec]
pool["specialisation_source"] = [src for _, src in spec]
pool = pool.join(hub_meta[["config_ok", "model_type", "arch_fingerprint", "context_window",
                           "params_b_hub", "weights_gb", "weights_dtypes", "hidden_size", "num_hidden_layers"]])
for df in energy_links.values():
    pool = pool.join(df)
pool = pool.join(openllm).join(mmlupro_tiger).join(mmlupro_oll).join(evalplus).join(prices)

# best available parameter count: Hub metadata, then ML.ENERGY, then the Open LLM Leaderboard
pool["params_b"] = (pool["params_b_hub"]
                    .fillna(pool.get("mle_total_params_billions"))
                    .fillna(pool["oll_params_b"].where(pool["oll_params_b"] > 0)))
pool["params_active_b"] = pool.get("mle_activated_params_billions").fillna(pool["params_b"])

# GPUs needed (wide table only): next power of two of weights/usable memory
gpus_raw = np.ceil(pool["weights_gb"] / (GPU_MEMORY_GB * WEIGHT_SHARE))
pool["gpus_needed_est"] = 2 ** np.ceil(np.log2(gpus_raw.clip(lower=1)))

quality_cols = [c for c in pool.columns if c.startswith(("oll_ifeval", "oll_bbh", "oll_math", "oll_gpqa",
                                                         "oll_musr", "oll_mmlu", "mmlupro_tiger_overall",
                                                         "evalplus_hum", "evalplus_mbpp"))]
link_cols = [f"{src}_link" for src in ENERGY_SOURCES]
pool["has_energy"] = pool[link_cols].notna().any(axis=1)
pool["energy_direct"] = (pool[link_cols] == "direct").any(axis=1)
pool["in_core"] = pool.index.isin(list(CORE_POOL))
pool["has_quality"] = pool[quality_cols].notna().any(axis=1)
pool["has_price"] = pool["price_out_usd_per_mtok"].notna()
pool["complete"] = pool["has_energy"] & pool["has_quality"]
pool = pool.reset_index()
print(pool[["has_energy", "energy_direct", "has_quality", "has_price", "complete", "in_core"]].sum().to_string())

has_energy        98
energy_direct     31
has_quality      157
has_price         39
complete          77
in_core            9


## Checks

In [18]:
assert pool["model_key"].is_unique
assert pool["specialisation"].notna().all()
assert not ((pool["specialisation"] == "medicine") & pool["model_key"].str.contains("medium")).any()
assert pool["mmlupro_tiger_overall"].dropna().between(0, 1).all(), "MMLU-Pro (TIGER-Lab) is on a 0-1 scale"
assert pool["model_key"].map(lambda k: k not in RENAMED).all(), "an old repository name is still in the pool"
for src in ENERGY_SOURCES:
    linked = pool[f"{src}_link"].notna()
    assert pool.loc[linked, f"{src}_donor"].notna().all(), f"{src}: linked rows without donor"
inherited = pool["llmperf_link"] == "inherited"
same_fp = pool.loc[inherited].apply(
    lambda r: fingerprint.get(r["llmperf_donor"]) == r["arch_fingerprint"], axis=1)
assert same_fp.all(), "an inherited value does not share the architecture fingerprint"
assert (pool["llmperf_j_per_in_token"].dropna() > 0).all()
print("All checks passed.")

All checks passed.


## Core pool and consistency checks

Energy from ML.ENERGY only, quality from the Open LLM Leaderboard only; the checks stop the notebook if a core model breaks these rules.

In [ ]:
core_cols = ["model_key", "specialisation", "params_b", "weights_gb", "gpus_needed_est", "mle_num_gpus",
             "context_window", "mle_link", "mle_donor", "mle_j_per_out_token", "mle_b8_j_per_out_token",
             "mle_avg_output_len", "mle_output_throughput_tokens_per_sec", "mle_avg_power_watts",
             "oll_ifeval", "oll_bbh", "oll_math_lvl_5", "oll_gpqa", "oll_musr", "oll_mmlu_pro",
             "price_in_usd_per_mtok", "price_out_usd_per_mtok", "arch_fingerprint"]
core = pool[pool["model_key"].isin(list(CORE_CANDIDATES))].copy()
core["core_role"] = core["model_key"].map(CORE_CANDIDATES)
core = core[["core_role"] + [c for c in core_cols if c in core] + [c for c in core if c.startswith("mmlupro_oll_")]]

# optional members: kept only if every value the pipeline needs is there
OLL6 = ["oll_ifeval", "oll_bbh", "oll_math_lvl_5", "oll_gpqa", "oll_musr", "oll_mmlu_pro"]
subject_cols = [c for c in core if c.startswith("mmlupro_oll_")]
def optional_gaps(r):
    gaps = []
    if pd.isna(r.get("mle_j_per_out_token")): gaps.append("ML.ENERGY energy")
    if r[OLL6].isna().any(): gaps.append("Open LLM scores")
    if not subject_cols or r[subject_cols].isna().all(): gaps.append("MMLU-Pro per subject")
    if pd.isna(r.get("context_window")): gaps.append("context window")
    if pd.isna(r.get("weights_gb")) and pd.isna(r.get("params_b")): gaps.append("size")
    return gaps
optional_status = {}
for m in CORE_OPTIONAL:
    row = core[core["model_key"] == m]
    optional_status[m] = ["not in the pool"] if row.empty else optional_gaps(row.iloc[0])
for m, gaps in optional_status.items():
    print(f"Optional core member {m}: " + ("included" if not gaps else "left out, missing " + ", ".join(gaps)))
core = core[~core["model_key"].isin([m for m, g in optional_status.items() if g])]
pool["in_core"] = pool["model_key"].isin(core["model_key"])

missing = set(CORE_POOL) - set(core["model_key"])
assert not missing, f"core models missing from the pool: {missing}"
assert core["mle_j_per_out_token"].notna().all(), "a core model has no ML.ENERGY energy:\n" + core.loc[core["mle_j_per_out_token"].isna(), "model_key"].to_string()
assert core[OLL6].notna().all().all(), "a core model lacks an Open LLM score:\n" + core.loc[core[OLL6].isna().any(axis=1), "model_key"].to_string()
gen_fp = set(core.loc[core["core_role"] == "generalist", "arch_fingerprint"])
assert core.loc[core["core_role"] != "generalist", "arch_fingerprint"].isin(gen_fp).all(), \
    "a core specialist has no generalist with the same architecture"
assert core["mle_donor"].map(key).isin(mle_curve["model_key"]).all(), "an energy donor of the core has no batch curve"
print(f"Core pool checks passed: {len(core)} models.")
print(core.drop(columns="arch_fingerprint").set_index("model_key").round(3).T.to_string())

Optional core member aaditya/llama3-openbiollm-8b: left out, missing Open LLM scores, MMLU-Pro per subject
Optional core member m42-health/llama3-med42-8b: left out, missing Open LLM scores, MMLU-Pro per subject
Core pool checks passed: 9 models.
model_key                                 aaditya/llama3-openbiollm-70b        m42-health/llama3-med42-70b  meta-llama/llama-3.1-70b-instruct  meta-llama/llama-3.1-8b-instruct  meta-llama/llama-3.3-70b-instruct meta-llama/meta-llama-3-70b-instruct meta-llama/meta-llama-3-8b-instruct nvidia/llama-3.1-nemotron-70b-instruct-hf      nvidia/openmath2-llama3.1-8b
core_role                                                      medicine                           medicine                         generalist                        generalist                         generalist                           generalist                          generalist                                generalist                       mathematics
specialisation                   

## Save

In [ ]:
pool_path = os.path.join(OUTPUT_DIR, f"model_pool_{POOL_VERSION}.parquet")
pool.to_parquet(pool_path, index=False)
core.to_parquet(os.path.join(OUTPUT_DIR, f"model_pool_core_{POOL_VERSION}.parquet"), index=False)
mle_curve.to_parquet(os.path.join(OUTPUT_DIR, f"mlenergy_batch_curve_{POOL_VERSION}.parquet"), index=False)

# raw per-source tables for analysis and reproducibility
for name, df in {"mlenergy": mlenergy, "llmperf": llmperf, "aienergy": aienergy,
                 "openllm": openllm, "mmlupro_tiger": mmlupro_tiger, "mmlupro_oll": mmlupro_oll, "evalplus": evalplus,
                 "prices": prices, "hub_meta": hub_meta}.items():
    df.reset_index().to_parquet(os.path.join(SNAPSHOT_DIR, f"{name}.parquet"), index=False)

metadata = {
    "pool_version": POOL_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "price_snapshot_utc": price_snapshot_utc,
    "sources": {
        "mlenergy": {"package": "mlenergy-data", "gpu": MLENERGY_GPU, "tasks": MLENERGY_TASKS, "batches": MLENERGY_BATCHES,
                     "curve_reference_batch": MLENERGY_REF_BATCH, "curve_file": f"mlenergy_batch_curve_{POOL_VERSION}.parquet"},
        "llmperf": {"repo": LLMPERF_REPO, "files": LLMPERF_FILES,
                    "revision": api.dataset_info(LLMPERF_REPO).sha},
        "aienergy": {"repo": AIENERGY_REPO, "file": AIENERGY_FILE, "unit": "kWh per 1,000 queries",
                     "campaigns": sorted(aies_by_campaign)},
        "mmlupro_oll_details": {"core_models_loaded": sorted(mmlupro_oll.index), "missing": details_missing},
        "openllm": {"repo": OLL_REPO, "revision": api.dataset_info(OLL_REPO).sha},
        "mmlupro_tiger": {"repo": TIGER_REPO, "file": TIGER_FILE, "revision": api.dataset_info(TIGER_REPO).sha,
                          "aliases": TIGER_ALIASES, "unmatched_names": unmatched},
        "evalplus": {"url": EVALPLUS_URL},
        "openrouter": {"url": OPENROUTER_URL},
        "software_paper": {"url": SOFTWARE_PAPER_ZIP, "merged": False},
    },
    "membership": {"official_orgs": sorted(OFFICIAL_ORGS), "instruct_regex": INSTRUCT_RE.pattern,
                   "exclude_regex": EXCLUDE_RE.pattern, "specialists": SPECIALISTS},
    "arch_fingerprint_keys": ARCH_KEYS,
    "renamed_ids": RENAMED,
    "core_pool": CORE_POOL,
    "core_optional": {m: ("included" if not g else "missing: " + ", ".join(g)) for m, g in optional_status.items()},
    "gpu_estimate": {"gpu_memory_gb": GPU_MEMORY_GB, "weight_share": WEIGHT_SHARE, "rounding": "next power of two"},
    "counts": {"members": len(pool),
               **{k: int(pool[k].sum()) for k in ["has_energy", "energy_direct", "has_quality", "has_price", "complete", "in_core"]}},
}
with open(os.path.join(OUTPUT_DIR, f"model_pool_{POOL_VERSION}_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2, default=str)
print("Saved:", pool_path)

Saved: /content/drive/MyDrive/thesis/model_pool/model_pool_v1.3.parquet
